# Retrieval-Augmented Generation (RAG) Pipeline

**Version:** 0.1.0-beta  
**Last Updated:** February 18, 2026 - 7:55 AM ET  
**Duration:** 50-60 minutes

Learn to build production RAG systems:
1. Azure AI Search integration
2. Hybrid retrieval (vector + keyword)
3. Document chunking strategies
4. Citation generation
5. Complete RAG pipeline
6. Quality evaluation

## Setup

In [ ]:
# Imports
import os
import sys
import asyncio
import json
from pathlib import Path
from typing import List, Dict, Any

# Setup paths
foundry_path = Path.cwd().parent
sys.path.insert(0, str(foundry_path))

# Load environment
from dotenv import load_dotenv
load_dotenv(foundry_path / ".env")

# Azure authentication
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()

print("✅ Setup complete")

## 1. Azure AI Search Setup

Connect to Azure AI Search service.

In [ ]:
from tools.search import EVASearchClient

# Initialize search client
search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
index_name = os.getenv("AZURE_SEARCH_INDEX", "ei-documents")

if not search_endpoint:
    print("❌ AZURE_SEARCH_ENDPOINT not configured in .env")
else:
    search_client = EVASearchClient(
        endpoint=search_endpoint,
        credential=credential,
        index_name=index_name
    )
    print(f"✅ Connected to Azure Search: {index_name}")

In [ ]:
# Test simple keyword search
async def test_keyword_search():
    results = await search_client.keyword_search(
        query="eligibility hours",
        top_k=3
    )
    
    print("🔍 KEYWORD SEARCH RESULTS:\n")
    for i, doc in enumerate(results, 1):
        title = doc.get("title", "Untitled")
        score = doc.get("@search.score", 0)
        content = doc.get("content", "")[:150]
        
        print(f"{i}. {title} (Score: {score:.2f})")
        print(f"   {content}...\n")

await test_keyword_search()

## 2. Vector Search

Semantic search using embeddings.

In [ ]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI

# Setup OpenAI client for embeddings
openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-15-preview"
)

# Get embedding for query
def get_embedding(text: str) -> List[float]:
    """Generate embedding vector for text."""
    response = openai_client.embeddings.create(
        input=text,
        model="text-embedding-ada-002"
    )
    return response.data[0].embedding

# Test embedding generation
query = "minimum hours required for EI"
query_vector = get_embedding(query)

print(f"✅ Generated embedding: {len(query_vector)} dimensions")
print(f"   First 5 values: {query_vector[:5]}")

In [ ]:
# Perform vector search
async def test_vector_search():
    results = await search_client.vector_search(
        query="explain EI eligibility requirements",
        top_k=3
    )
    
    print("🧭 VECTOR SEARCH RESULTS (Semantic):\n")
    for i, doc in enumerate(results, 1):
        title = doc.get("title", "Untitled")
        similarity = doc.get("@search.score", 0)
        content = doc.get("content", "")[:150]
        
        print(f"{i}. {title} (Similarity: {similarity:.3f})")
        print(f"   {content}...\n")

await test_vector_search()

## 3. Hybrid Search

Combine keyword + vector + semantic reranking.

In [ ]:
# Perform hybrid search
async def test_hybrid_search():
    results = await search_client.hybrid_search(
        query="What documentation is needed to apply for EI benefits?",
        top_k=5
    )
    
    print("⚡ HYBRID SEARCH RESULTS (Keyword + Vector + Reranking):\n")
    for i, doc in enumerate(results, 1):
        title = doc.get("title", "Untitled")
        section = doc.get("section", "N/A")
        score = doc.get("@search.score", 0)
        reranker_score = doc.get("@search.rerankerScore", None)
        content = doc.get("content", "")[:200]
        
        print(f"{i}. {title}")
        print(f"   Section: {section}")
        print(f"   Scores: Search={score:.2f}, Reranker={reranker_score:.2f if reranker_score else 'N/A'}")
        print(f"   {content}...\n")

await test_hybrid_search()

## 4. Document Chunking

Split documents for optimal retrieval.

In [ ]:
from tools.rag import DocumentChunker

# Sample document
sample_doc = """
Employment Insurance (EI) provides temporary financial assistance to unemployed Canadians 
while they look for work or upgrade their skills.

To qualify for EI regular benefits, you need:
- Between 420 and 700 insurable hours in the last 52 weeks (varies by region)
- Your employment was interrupted through no fault of your own
- You have been without work and pay for at least 7 consecutive days
- You are ready, willing, and capable of working each day
- You are actively looking for work

The amount of EI you receive depends on your earnings in the qualifying period. 
The basic benefit rate is 55% of your average weekly insurable earnings, up to a maximum 
of $668 per week (as of 2026).

You must apply for EI benefits as soon as you stop working. You can apply online through 
your My Service Canada Account. You will need your Social Insurance Number (SIN) and the 
Record of Employment (ROE) from all your employers in the last 52 weeks.
"""

# Initialize chunker
chunker = DocumentChunker(
    chunk_size=512,  # tokens
    chunk_overlap=50  # token overlap between chunks
)

# Create chunks
chunks = chunker.chunk_text(sample_doc)

print(f"📄 Original document: {len(sample_doc)} characters")
print(f"✂️  Created {len(chunks)} chunks:\n")

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i} ({len(chunk)} chars):")
    print(f"{chunk[:150]}...\n")

In [ ]:
# Semantic chunking (chunk by meaning, not just size)
semantic_chunks = chunker.chunk_by_sentences(
    text=sample_doc,
    max_sentences=3  # Group related sentences
)

print(f"🧠 SEMANTIC CHUNKING ({len(semantic_chunks)} chunks):\n")

for i, chunk in enumerate(semantic_chunks, 1):
    print(f"Chunk {i}:")
    print(f"{chunk.strip()}\n")

## 5. Citation Generation

Build proper legal citations.

In [ ]:
from tools.rag import CitationBuilder

# Initialize citation builder (McGill Guide format)
citation_builder = CitationBuilder(style="mcgill")

# Sample sources
sources = [
    {
        "type": "statute",
        "title": "Employment Insurance Act",
        "jurisdiction": "SC",
        "year": "1996",
        "chapter": "c 23",
        "section": "7(2)"
    },
    {
        "type": "regulation",
        "title": "Employment Insurance Regulations",
        "jurisdiction": "SOR",
        "year": "1996",
        "number": "332",
        "section": "54"
    },
    {
        "type": "case",
        "case_name": "Canada (Attorney General) v Smith",
        "year": "2023",
        "court": "FCA",
        "citation": "2023 FCA 45",
        "paragraph": "28"
    }
]

# Build citations
print("📚 FORMATTED CITATIONS (McGill Guide):\n")
for i, source in enumerate(sources, 1):
    citation = citation_builder.format_citation(source)
    print(f"{i}. {citation}\n")

In [ ]:
# Extract citations from search results
async def extract_citations_from_search():
    results = await search_client.hybrid_search(
        query="EI benefit calculation formula",
        top_k=3
    )
    
    citations = []
    for doc in results:
        citation_data = {
            "type": doc.get("document_type", "document"),
            "title": doc.get("title", "Untitled"),
            "section": doc.get("section"),
            "url": doc.get("url"),
            "page": doc.get("page_number")
        }
        citations.append(citation_builder.format_citation(citation_data))
    
    return citations

search_citations = await extract_citations_from_search()
print("📖 CITATIONS FROM SEARCH:\n")
for i, cite in enumerate(search_citations, 1):
    print(f"{i}. {cite}")

## 6. Complete RAG Pipeline

Combine retrieval, generation, and citation.

In [ ]:
from tools.rag import RAGPipeline
from agent_framework import Agent

# Initialize RAG pipeline
rag_pipeline = RAGPipeline(
    search_client=search_client,
    chunker=chunker,
    citation_builder=citation_builder,
    top_k=5,
    rerank=True
)

# Create RAG-enabled agent
rag_agent = Agent(
    name="RAGAssistant",
    instructions="""You are an EI policy expert. Use the provided context to answer questions accurately.
    
    Always:
    - Base answers on provided context
    - Cite sources using the citation format provided
    - Be precise and specific
    - Acknowledge if information is not in context
    """,
    model="gpt-4o",
    temperature=0.3
)

print("✅ RAG pipeline initialized")

In [ ]:
# Execute RAG pipeline
async def run_rag_query(question: str):
    print(f"❓ QUESTION: {question}\n")
    
    # Step 1: Retrieve relevant documents
    print("🔍 Step 1: Retrieving documents...")
    retrieved_docs = await rag_pipeline.retrieve(question)
    print(f"   Found {len(retrieved_docs)} relevant documents\n")
    
    # Step 2: Build context from chunks
    print("📝 Step 2: Building context...")
    context = rag_pipeline.build_context(retrieved_docs)
    print(f"   Context: {len(context)} characters\n")
    
    # Step 3: Generate answer with agent
    print("🤖 Step 3: Generating answer...")
    prompt = f"""Context:
{context}

Question: {question}

Provide a detailed answer based on the context above. Include citations."""
    
    response = await rag_agent.run(prompt, thread_id="rag-demo")
    print(f"   Generated response\n")
    
    # Step 4: Extract and format citations
    print("📚 Step 4: Formatting citations...")
    citations = rag_pipeline.extract_citations(retrieved_docs)
    print(f"   Created {len(citations)} citations\n")
    
    return {
        "question": question,
        "answer": response.content,
        "sources": retrieved_docs,
        "citations": citations
    }

# Test RAG pipeline
result = await run_rag_query(
    "How many hours do I need to qualify for EI in a region with 10% unemployment?"
)

print("="*80)
print("\n💡 ANSWER:")
print(result["answer"])
print("\n📖 SOURCES:")
for i, cite in enumerate(result["citations"], 1):
    print(f"{i}. {cite}")

## 7. Batch RAG Processing

Process multiple questions efficiently.

In [ ]:
# Batch questions
questions = [
    "What is the maximum weekly EI benefit in 2026?",
    "Can I receive EI if I quit my job?",
    "How long do I have to apply for EI after job loss?",
    "What is the waiting period for EI benefits?"
]

async def batch_rag_processing(questions: List[str]):
    results = []
    
    for i, q in enumerate(questions, 1):
        print(f"Processing {i}/{len(questions)}: {q[:50]}...")
        result = await run_rag_query(q)
        results.append(result)
        print("")
    
    return results

batch_results = await batch_rag_processing(questions)

print("\n" + "="*80)
print("📊 BATCH PROCESSING RESULTS\n")

for i, result in enumerate(batch_results, 1):
    print(f"{i}. Q: {result['question']}")
    print(f"   A: {result['answer'][:200]}...\n")

## 8. RAG Quality Evaluation

Measure retrieval and generation quality.

In [ ]:
from tools.evaluation import EVAEvaluator

# Initialize evaluator
evaluator = EVAEvaluator(
    model="gpt-4o",
    deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT")
)

async def evaluate_rag_response(result: Dict[str, Any]):
    # Build source context
    source_context = "\n\n".join([
        doc.get("content", "") for doc in result["sources"][:3]
    ])
    
    # Evaluate groundedness (answer based on sources?)
    groundedness = await evaluator.evaluate_groundedness(
        query=result["question"],
        response=result["answer"],
        context=source_context
    )
    
    # Evaluate relevance (answer addresses question?)
    relevance = await evaluator.evaluate_relevance(
        query=result["question"],
        response=result["answer"]
    )
    
    # Evaluate coherence (answer well-structured?)
    coherence = await evaluator.evaluate_coherence(
        response=result["answer"]
    )
    
    return {
        "groundedness": groundedness,
        "relevance": relevance,
        "coherence": coherence,
        "average": (groundedness + relevance + coherence) / 3
    }

# Evaluate first result
eval_scores = await evaluate_rag_response(result)

print("📊 RAG QUALITY METRICS:\n")
print(f"Groundedness: {eval_scores['groundedness']:.2f} (answer based on sources)")
print(f"Relevance:    {eval_scores['relevance']:.2f} (answer addresses question)")
print(f"Coherence:    {eval_scores['coherence']:.2f} (answer well-structured)")
print(f"\n📈 Overall Score: {eval_scores['average']:.2f}/5.0")

if eval_scores['average'] >= 4.0:
    print("✅ Excellent quality")
elif eval_scores['average'] >= 3.0:
    print("⚠️  Good quality, some improvements possible")
else:
    print("❌ Needs improvement")

## 9. Optimization: Reranking Strategies

Improve retrieval quality with reranking.

In [ ]:
# Compare search strategies
async def compare_search_strategies(query: str):
    strategies = [
        ("Keyword Only", search_client.keyword_search),
        ("Vector Only", search_client.vector_search),
        ("Hybrid", search_client.hybrid_search)
    ]
    
    print(f"🔍 Query: {query}\n")
    print("="*80 + "\n")
    
    for name, search_func in strategies:
        results = await search_func(query=query, top_k=3)
        
        print(f"📊 {name.upper()}:")
        for i, doc in enumerate(results, 1):
            title = doc.get("title", "Untitled")[:40]
            score = doc.get("@search.score", 0)
            print(f"   {i}. {title}... (Score: {score:.3f})")
        print("")

await compare_search_strategies(
    "What are the consequences of refusing suitable employment while on EI?"
)

## 10. Summary

### What You've Learned

1. ✅ **Azure AI Search:** Keyword, vector, and hybrid search
2. ✅ **Document Chunking:** Fixed-size and semantic chunking strategies
3. ✅ **Citation Generation:** McGill Guide formatting for legal sources
4. ✅ **RAG Pipeline:** Complete retrieve→generate→cite workflow
5. ✅ **Batch Processing:** Efficiently handle multiple queries
6. ✅ **Quality Evaluation:** Groundedness, relevance, coherence metrics
7. ✅ **Optimization:** Reranking and search strategy comparison

### Key Metrics for Production

- **Groundedness:** Ensure answers are based on retrieved sources (target: >4.0)
- **Relevance:** Answers address the specific question (target: >4.0)
- **Retrieval Quality:** Top-3 documents contain answer (target: >90%)
- **Citation Accuracy:** All claims properly cited (target: 100%)

### Next Steps

- **Notebook 04:** Comprehensive evaluation and A/B testing
- **Production:** Deploy RAG pipeline to Azure AI Foundry
- **Monitoring:** Track quality metrics in production

## 11. Cleanup

In [ ]:
# Close connections
await search_client.close()

print("🎉 RAG pipeline tutorial complete!")
print("\n📚 You can now build production-grade retrieval systems.")